# Scene Weaver — GPU encoder (Google Colab T4)

1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. Copy the **https link** printed at the bottom and paste it into the app's *Colab GPU encoder* box.

Keep this tab open while the video renders.

In [ ]:
#@title 2 · Start the encoder + public https tunnel
APP_URL = "https://epic-colab-maker.lovable.app"  #@param {type:"string"}
TOKEN   = ""  #@param {type:"string"}

import os, re, base64, subprocess, threading, time, urllib.request
os.environ['SW_TOKEN'] = TOKEN

# download latest encoder from the live app; fall back to the embedded copy
try:
    req = urllib.request.Request(APP_URL.rstrip('/') + '/colab/encoder_server.py',
                                 headers={'User-Agent': 'scene-weaver-colab'})
    with urllib.request.urlopen(req, timeout=30) as r, open('/content/encoder_server.py', 'wb') as f:
        f.write(r.read())
    print('encoder downloaded from', APP_URL)
except Exception as e:
    print('download failed ({e}) — using embedded encoder copy')
    _B64 = "IiIiClNjZW5lIFdlYXZlciDigJQgQ29sYWIgVDQgR1BVIGVuY29kZXIuCgpSdW5zIGluc2lkZSBhIEdvb2dsZSBDb2xhYiBub3RlYm9vayAoR1BVIHJ1bnRpbWUpLiBBY2NlcHRzIGEgcGFuZWwgbGlzdCBmcm9tCnRoZSB3ZWIgYXBwLCBkb3dubG9hZHMgZXZlcnkgcGFuZWwgaW1hZ2UsIHJlbmRlcnMgS2VuIEJ1cm5zIG1vdGlvbiArIGNvbG91cgpncmFkaW5nIHdpdGggZmZtcGVnLCBlbmNvZGVzIHdpdGggdGhlIFQ0J3MgTlZFTkMgaGFyZHdhcmUgZW5jb2RlciBhbmQgc2VydmVzCnRoZSBmaW5pc2hlZCBtcDQgYmFjayBvdmVyIGFuIGh0dHBzIHR1bm5lbC4KCkRlc2lnbiBub3RlcyBmb3IgdmVyeSBsb25nIHZpZGVvcyAoMmgrLCB0aG91c2FuZHMgb2YgcGFuZWxzKToKICAqIEVhY2ggcGFuZWwgYmVjb21lcyBpdHMgb3duIHNob3J0IGNsaXAgLT4gbWVtb3J5IHN0YXlzIGZsYXQuCiAgKiBDbGlwcyBhcmUgY3Jvc3MtZmFkZWQgaW4gZ3JvdXBzIChHUk9VUCBwYW5lbHMgcGVyIGZpbHRlcl9jb21wbGV4KSBzbyB0aGUKICAgIGZmbXBlZyBjb21tYW5kIG5ldmVyIGdyb3dzIHVuYm91bmRlZCwgdGhlbiB0aGUgZ3JvdXBzIGFyZSBzdHJlYW0tY29weQogICAgY29uY2F0ZW5hdGVkOiBubyBnZW5lcmF0aW9uIGxvc3MsIG5vIE8obl4yKSByZS1lbmNvZGluZy4KICAqIFBhbmVscyBhcmUgcmVuZGVyZWQgaW4gcGFyYWxsZWwgbGFuZXM7IE5WRU5DIG9uIGEgVDQgaGFuZGxlcyBzZXZlcmFsCiAgICAxMDgwcDMwIHN0cmVhbXMgYXQgb25jZS4KIiIiCgppbXBvcnQganNvbiwgbWF0aCwgb3MsIHJlLCBzaHV0aWwsIHN1YnByb2Nlc3MsIHRocmVhZGluZywgdGltZSwgdXVpZCwgaGFzaGxpYgpmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgVGhyZWFkUG9vbEV4ZWN1dG9yCmZyb20gaHR0cC5zZXJ2ZXIgaW1wb3J0IEJhc2VIVFRQUmVxdWVzdEhhbmRsZXIsIFRocmVhZGluZ0hUVFBTZXJ2ZXIKZnJvbSB1cmxsaWIucGFyc2UgaW1wb3J0IHVybHBhcnNlCmltcG9ydCB1cmxsaWIucmVxdWVzdAoKVywgSCwgRlBTLCBYRiwgR1JPVVAgPSAxOTIwLCAxMDgwLCAzMCwgMC43LCA0MApXT1JLID0gIi9jb250ZW50L3N3X3dvcmsiCk9VVCA9ICIvY29udGVudC9zd19vdXQiCkxBTkVTID0gaW50KG9zLmVudmlyb24uZ2V0KCJTV19MQU5FUyIsICI0IikpClRPS0VOID0gb3MuZW52aXJvbi5nZXQoIlNXX1RPS0VOIiwgIiIpCgpvcy5tYWtlZGlycyhXT1JLLCBleGlzdF9vaz1UcnVlKQpvcy5tYWtlZGlycyhPVVQsIGV4aXN0X29rPVRydWUpCgpKT0JTID0ge30KTE9DSyA9IHRocmVhZGluZy5Mb2NrKCkKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBlbmNvZGVyIC0tLQoKZGVmIGhhc19udmVuYygpOgogICAgdHJ5OgogICAgICAgIG91dCA9IHN1YnByb2Nlc3MucnVuKFsiZmZtcGVnIiwgIi1oaWRlX2Jhbm5lciIsICItZW5jb2RlcnMiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUpLnN0ZG91dAogICAgICAgIHJldHVybiAiaDI2NF9udmVuYyIgaW4gb3V0CiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBGYWxzZQoKTlZFTkMgPSBoYXNfbnZlbmMoKQpWQ09ERUMgPSAoWyItYzp2IiwgImgyNjRfbnZlbmMiLCAiLXByZXNldCIsICJwNCIsICItcmMiLCAidmJyIiwgIi1jcSIsICIyMyIsICItYjp2IiwgIjhNIl0KICAgICAgICAgIGlmIE5WRU5DIGVsc2UKICAgICAgICAgIFsiLWM6diIsICJsaWJ4MjY0IiwgIi1wcmVzZXQiLCAidmVyeWZhc3QiLCAiLWNyZiIsICIyMSJdKQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY2luZW1hdG9ncmFwaHkgLS0tCgpNT1ZFUyA9IFsKICAgICgxLjAwLCAxLjIwLCAwLjUsIDAuNSwgMC41LCAwLjUpLAogICAgKDEuMjIsIDEuMDAsIDAuNSwgMC41LCAwLjUsIDAuNSksCiAgICAoMS4xNCwgMS4xNCwgMC4wLCAxLjAsIDAuNSwgMC41KSwKICAgICgxLjE0LCAxLjE0LCAxLjAsIDAuMCwgMC41LCAwLjUpLAogICAgKDEuMTQsIDEuMTQsIDAuNSwgMC41LCAwLjAsIDEuMCksCiAgICAoMS4xNCwgMS4xNCwgMC41LCAwLjUsIDEuMCwgMC4wKSwKICAgICgxLjAyLCAxLjI0LCAwLjMsIDAuMjgsIDAuMywgMC4yMiksCiAgICAoMS4wMiwgMS4yNCwgMC43LCAwLjcyLCAwLjcsIDAuNzgpLAogICAgKDEuMDgsIDEuMjIsIDAuMTUsIDAuODUsIDAuODUsIDAuMTUpLApdCgpHUkFERVMgPSB7CiAgICAibmlnaHQiOiAgKCIxLjE0IiwgIi0wLjA0NSIsICIxLjA1IiwgIjAuMTA6MC4wMjotMC4xMCIpLAogICAgInN1bnNldCI6ICgiMS4xMCIsICIwLjAyIiwgIjEuMzAiLCAiMC4xMDowLjAxOi0wLjA4IiksCiAgICAid2FybSI6ICAgKCIxLjA4IiwgIjAuMDE1IiwgIjEuMjIiLCAiMC4wNjowLjAwOi0wLjA1IiksCiAgICAiY29vbCI6ICAgKCIxLjEwIiwgIjAuMCIsICIxLjEyIiwgIi0wLjA2OjAuMDA6MC4wNyIpLAogICAgInRlbnNlIjogICgiMS4yNiIsICItMC4wMyIsICIwLjkyIiwgIjAuMDc6LTAuMDI6LTAuMDMiKSwKICAgICJyYWluIjogICAoIjEuMTIiLCAiLTAuMDIiLCAiMC45NSIsICItMC4wNTowLjAwOjAuMDgiKSwKICAgICJicmlnaHQiOiAoIjEuMDYiLCAiMC4wMzUiLCAiMS4yOCIsICIwLjAzOjAuMDE6LTAuMDIiKSwKICAgICJkcmVhbSI6ICAoIjEuMDIiLCAiMC4wMyIsICIxLjM0IiwgIjAuMDU6LTAuMDE6MC4wNSIpLAp9CkNZQ0xFID0gWyJicmlnaHQiLCAid2FybSIsICJjb29sIiwgImRyZWFtIiwgInRlbnNlIl0KCktFWVMgPSBbCiAgICAoIm5pZ2h0IiwgWyJuaWdodCIsICJtaWRuaWdodCIsICJtb29uIiwgImRhcmsgcm9vbSIsICJzdGFybGl0IiwgInN0cmVldGxpZ2h0Il0pLAogICAgKCJzdW5zZXQiLCBbInN1bnNldCIsICJkdXNrIiwgImdvbGRlbiBob3VyIiwgInN1bnJpc2UiLCAiZGF3biIsICJmaXJlIiwgImZsYW1lIiwgImxhbnRlcm4iXSksCiAgICAoInJhaW4iLCBbInJhaW4iLCAic3Rvcm0iLCAid2V0IiwgIm1vbnNvb24iLCAiZm9nIiwgIm1pc3QiXSksCiAgICAoInRlbnNlIiwgWyJhbmdyeSIsICJmaWdodCIsICJibG9vZCIsICJzY3JlYW0iLCAiZmVhciIsICJzaGFkb3ciLCAidGhyZWF0IiwgImJhdHRsZSJdKSwKICAgICgiYnJpZ2h0IiwgWyJzdW5saWdodCIsICJzdW5ueSIsICJtb3JuaW5nIiwgIm1hcmtldCIsICJmZXN0aXZhbCIsICJzbWlsZSIsICJsYXVnaCJdKSwKICAgICgiZHJlYW0iLCBbIm1lbW9yeSIsICJkcmVhbSIsICJmbGFzaGJhY2siLCAic2t5IiwgImhvcGUiLCAibWFnaWMiXSksCiAgICAoIndhcm0iLCBbImluZG9vciIsICJyb29tIiwgImtpdGNoZW4iLCAibGFtcCIsICJ3YXJtIl0pLAogICAgKCJjb29sIiwgWyJjb2xkIiwgInJvb2Z0b3AiLCAiaG9zcGl0YWwiLCAib2ZmaWNlIiwgInNjaG9vbCIsICJ0cmFpbiJdKSwKXQoKCmRlZiBncmFkZV9mb3IocHJvbXB0LCBpKToKICAgIHQgPSAocHJvbXB0IG9yICIiKS5sb3dlcigpCiAgICBmb3IgbmFtZSwgd29yZHMgaW4gS0VZUzoKICAgICAgICBpZiBhbnkodyBpbiB0IGZvciB3IGluIHdvcmRzKToKICAgICAgICAgICAgcmV0dXJuIEdSQURFU1tuYW1lXQogICAgcmV0dXJuIEdSQURFU1tDWUNMRVtpICUgbGVuKENZQ0xFKV1dCgoKZGVmIG1vdmVfZm9yKGkpOgogICAgaCA9IGludChoYXNobGliLm1kNShmIm17aX0iLmVuY29kZSgpKS5oZXhkaWdlc3QoKVs6OF0sIDE2KQogICAgcmV0dXJuIE1PVkVTW2ggJSBsZW4oTU9WRVMpXQoKCmRlZiBjbGlwX2ZpbHRlcihpLCBkdXIsIHByb21wdCk6CiAgICAiIiJ6b29tcGFuIEtlbiBCdXJucyArIGNvbG91ciBncmFkZSwgYWx3YXlzIG91dHB1dCBleGFjdCAxNjo5IDEwODBwLiIiIgogICAgejAsIHoxLCB4MCwgeDEsIHkwLCB5MSA9IG1vdmVfZm9yKGkpCiAgICBmcmFtZXMgPSBtYXgoMSwgcm91bmQoZHVyICogRlBTKSkKICAgIGNvbnRyYXN0LCBicmlnaHQsIHNhdCwgY2IgPSBncmFkZV9mb3IocHJvbXB0LCBpKQogICAgIyBwcm9ncmVzcyAwLi4xIGFjcm9zcyB0aGUgY2xpcCwgZWFzZWQKICAgIHAgPSBmIihvbi97bWF4KDEsIGZyYW1lcyAtIDEpfSkiCiAgICBlID0gZiIoe3B9KntwfSooMy0yKntwfSkpIgogICAgeiA9IGYiKHt6MH0rKHt6MX0te3owfSkqe2V9KSIKICAgIGZ4ID0gZiIoe3gwfSsoe3gxfS17eDB9KSp7ZX0pIgogICAgZnkgPSBmIih7eTB9Kyh7eTF9LXt5MH0pKntlfSkiCiAgICByZXR1cm4gKAogICAgICAgIGYic2NhbGU9e1cqMn06e0gqMn06Zm9yY2Vfb3JpZ2luYWxfYXNwZWN0X3JhdGlvPWluY3JlYXNlLCIKICAgICAgICBmImNyb3A9e1cqMn06e0gqMn0sc2V0c2FyPTEsIgogICAgICAgIGYiem9vbXBhbj16PSd7en0nOng9Jyhpdy1pdy96b29tKSp7Znh9Jzp5PScoaWgtaWgvem9vbSkqe2Z5fSciCiAgICAgICAgZiI6ZD17ZnJhbWVzfTpzPXtXfXh7SH06ZnBzPXtGUFN9LCIKICAgICAgICBmImVxPWNvbnRyYXN0PXtjb250cmFzdH06YnJpZ2h0bmVzcz17YnJpZ2h0fTpzYXR1cmF0aW9uPXtzYXR9LCIKICAgICAgICBmImNvbG9yYmFsYW5jZT1ybT17Y2Iuc3BsaXQoJzonKVswXX06Z209e2NiLnNwbGl0KCc6JylbMV19OmJtPXtjYi5zcGxpdCgnOicpWzJdfSwiCiAgICAgICAgZiJmb3JtYXQ9eXV2NDIwcCIKICAgICkKCgpkZWYgcnVuKGNtZCk6CiAgICByID0gc3VicHJvY2Vzcy5ydW4oY21kLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUpCiAgICBpZiByLnJldHVybmNvZGUgIT0gMDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3Ioci5zdGRlcnJbLTE1MDA6XSkKCgpkZWYgZmV0Y2godXJsLCBwYXRoLCBhdHRlbXB0cz00KToKICAgIGxhc3QgPSAiIgogICAgZm9yIGEgaW4gcmFuZ2UoYXR0ZW1wdHMpOgogICAgICAgIHRyeToKICAgICAgICAgICAgcmVxID0gdXJsbGliLnJlcXVlc3QuUmVxdWVzdCh1cmwsIGhlYWRlcnM9eyJVc2VyLUFnZW50IjogInNjZW5lLXdlYXZlci1jb2xhYiJ9KQogICAgICAgICAgICB3aXRoIHVybGxpYi5yZXF1ZXN0LnVybG9wZW4ocmVxLCB0aW1lb3V0PTkwKSBhcyByLCBvcGVuKHBhdGgsICJ3YiIpIGFzIGY6CiAgICAgICAgICAgICAgICBzaHV0aWwuY29weWZpbGVvYmoociwgZikKICAgICAgICAgICAgaWYgb3MucGF0aC5nZXRzaXplKHBhdGgpID4gMDoKICAgICAgICAgICAgICAgIHJldHVybgogICAgICAgICAgICBsYXN0ID0gImVtcHR5IGZpbGUiCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICBsYXN0ID0gc3RyKGUpCiAgICAgICAgdGltZS5zbGVlcCgwLjYgKiAoYSArIDEpKQogICAgcmFpc2UgUnVudGltZUVycm9yKGYiZG93bmxvYWQgZmFpbGVkOiB7bGFzdH0iKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBqb2IgLS0tCgpkZWYgc2V0X2pvYihqaWQsICoqa3cpOgogICAgd2l0aCBMT0NLOgogICAgICAgIEpPQlNbamlkXS51cGRhdGUoa3cpCgoKZGVmIHJlbmRlcihqaWQsIHBhbmVscyk6CiAgICBkID0gb3MucGF0aC5qb2luKFdPUkssIGppZCkKICAgIG9zLm1ha2VkaXJzKGQsIGV4aXN0X29rPVRydWUpCiAgICBuID0gbGVuKHBhbmVscykKICAgIGRvbmUgPSBbMF0KCiAgICBkZWYgb25lKGkpOgogICAgICAgIHAgPSBwYW5lbHNbaV0KICAgICAgICBkdXIgPSBtYXgoMC44LCBmbG9hdChwWyJlbmQiXSkgLSBmbG9hdChwWyJzdGFydCJdKSkKICAgICAgICAjIGNyb3NzZmFkZSBuZWVkcyBYRiBleHRyYSBzZWNvbmRzIG9mIHRhaWwgb24gZXZlcnkgY2xpcCBidXQgdGhlIGxhc3QKICAgICAgICB0YWlsID0gWEYgaWYgaSA8IG4gLSAxIGVsc2UgMC4wCiAgICAgICAgaW1nID0gb3MucGF0aC5qb2luKGQsIGYiaXtpOjA2ZH0iKQogICAgICAgIGNsaXAgPSBvcy5wYXRoLmpvaW4oZCwgZiJje2k6MDZkfS5tcDQiKQogICAgICAgIGZldGNoKHBbInVybCJdLCBpbWcpCiAgICAgICAgcnVuKFsiZmZtcGVnIiwgIi15IiwgIi1sb29wIiwgIjEiLCAiLWkiLCBpbWcsICItdCIsIGYie2R1ciArIHRhaWw6LjNmfSIsCiAgICAgICAgICAgICAiLXZmIiwgY2xpcF9maWx0ZXIoaSwgZHVyICsgdGFpbCwgcC5nZXQoInByb21wdCIpKSwKICAgICAgICAgICAgICItciIsIHN0cihGUFMpLCAqVkNPREVDLCAiLXBpeF9mbXQiLCAieXV2NDIwcCIsIGNsaXBdKQogICAgICAgIG9zLnJlbW92ZShpbWcpCiAgICAgICAgZG9uZVswXSArPSAxCiAgICAgICAgc2V0X2pvYihqaWQsIHBjdD1yb3VuZChkb25lWzBdIC8gbiAqIDc4KSwKICAgICAgICAgICAgICAgIG5vdGU9ZiJSZW5kZXJpbmcgcGFuZWxzIG9uIEdQVSDCtyB7ZG9uZVswXX0ve259IikKICAgICAgICByZXR1cm4gZHVyCgogICAgd2l0aCBUaHJlYWRQb29sRXhlY3V0b3IobWF4X3dvcmtlcnM9TEFORVMpIGFzIGV4OgogICAgICAgIGR1cnMgPSBsaXN0KGV4Lm1hcChvbmUsIHJhbmdlKG4pKSkKCiAgICAjIC0tLS0gY3Jvc3MtZmFkZSBpbnNpZGUgZ3JvdXBzLCB0aGVuIHN0cmVhbS1jb3B5IGNvbmNhdCB0aGUgZ3JvdXBzIC0tLS0tLQogICAgZ3JvdXBzID0gW10KICAgIGdpID0gMAogICAgZm9yIGcwIGluIHJhbmdlKDAsIG4sIEdST1VQKToKICAgICAgICBpZHhzID0gbGlzdChyYW5nZShnMCwgbWluKG4sIGcwICsgR1JPVVApKSkKICAgICAgICBncGF0aCA9IG9zLnBhdGguam9pbihkLCBmImd7Z2k6MDVkfS5tcDQiKQogICAgICAgIGlmIGxlbihpZHhzKSA9PSAxOgogICAgICAgICAgICBzaHV0aWwuY29weShvcy5wYXRoLmpvaW4oZCwgZiJje2lkeHNbMF06MDZkfS5tcDQiKSwgZ3BhdGgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYXJncywgZmMsIHByZXYgPSBbXSwgW10sICIwOnYiCiAgICAgICAgICAgIGZvciBpIGluIGlkeHM6CiAgICAgICAgICAgICAgICBhcmdzICs9IFsiLWkiLCBvcy5wYXRoLmpvaW4oZCwgZiJje2k6MDZkfS5tcDQiKV0KICAgICAgICAgICAgIyBjbGlwIGsgc3RhcnRzIGF0IHRoZSBzdW0gb2YgdGhlICp2aXNpYmxlKiBkdXJhdGlvbnMgYmVmb3JlIGl0LAogICAgICAgICAgICAjIGFuZCBpdHMgY3Jvc3MtZmFkZSB3aXRoIHRoZSBwcmV2aW91cyBjbGlwIGJlZ2lucyBleGFjdGx5IHRoZXJlCiAgICAgICAgICAgIG9mZiA9IDAuMAogICAgICAgICAgICBmb3IgayBpbiByYW5nZSgxLCBsZW4oaWR4cykpOgogICAgICAgICAgICAgICAgb2ZmICs9IG1heCgwLjgsIGR1cnNbaWR4c1trIC0gMV1dKQogICAgICAgICAgICAgICAgbGFiID0gZiJ4e2t9IgogICAgICAgICAgICAgICAgZmMuYXBwZW5kKGYiW3twcmV2fV1be2t9OnZdeGZhZGU9dHJhbnNpdGlvbj1mYWRlOmR1cmF0aW9uPXtYRn06IgogICAgICAgICAgICAgICAgICAgICAgICAgIGYib2Zmc2V0PXttYXgoMC4wNSwgb2ZmKTouM2Z9W3tsYWJ9XSIpCiAgICAgICAgICAgICAgICBwcmV2ID0gbGFiCgogICAgICAgICAgICBzcGFuID0gc3VtKG1heCgwLjgsIGR1cnNbaV0pIGZvciBpIGluIGlkeHMpCiAgICAgICAgICAgIHJ1bihbImZmbXBlZyIsICIteSIsICphcmdzLCAiLWZpbHRlcl9jb21wbGV4IiwgIjsiLmpvaW4oZmMpLAogICAgICAgICAgICAgICAgICItbWFwIiwgZiJbe3ByZXZ9XSIsICItdCIsIGYie3NwYW46LjNmfSIsCiAgICAgICAgICAgICAgICAgIi1yIiwgc3RyKEZQUyksICpWQ09ERUMsICItcGl4X2ZtdCIsICJ5dXY0MjBwIiwgZ3BhdGhdKQoKICAgICAgICBncm91cHMuYXBwZW5kKGdwYXRoKQogICAgICAgIGdpICs9IDEKICAgICAgICBzZXRfam9iKGppZCwgcGN0PTc4ICsgcm91bmQoZ2kgLyBtYXgoMSwgbWF0aC5jZWlsKG4gLyBHUk9VUCkpICogMTgpLAogICAgICAgICAgICAgICAgbm90ZT1mIlN0aXRjaGluZyDCtyBwYXJ0IHtnaX0ve21hdGguY2VpbChuIC8gR1JPVVApfSIpCgogICAgbGlzdGYgPSBvcy5wYXRoLmpvaW4oZCwgImxpc3QudHh0IikKICAgIHdpdGggb3BlbihsaXN0ZiwgInciKSBhcyBmOgogICAgICAgIGZvciBnIGluIGdyb3VwczoKICAgICAgICAgICAgZi53cml0ZShmImZpbGUgJ3tnfSdcbiIpCiAgICBmaW5hbCA9IG9zLnBhdGguam9pbihPVVQsIGYie2ppZH0ubXA0IikKICAgIHNldF9qb2IoamlkLCBwY3Q9OTcsIG5vdGU9IldyaXRpbmcgZmluYWwgbXA04oCmIikKICAgIHJ1bihbImZmbXBlZyIsICIteSIsICItZiIsICJjb25jYXQiLCAiLXNhZmUiLCAiMCIsICItaSIsIGxpc3RmLAogICAgICAgICAiLWMiLCAiY29weSIsICItbW92ZmxhZ3MiLCAiK2Zhc3RzdGFydCIsIGZpbmFsXSkKICAgIHNodXRpbC5ybXRyZWUoZCwgaWdub3JlX2Vycm9ycz1UcnVlKQogICAgc2l6ZSA9IG9zLnBhdGguZ2V0c2l6ZShmaW5hbCkKICAgIHNldF9qb2IoamlkLCBwY3Q9MTAwLCBzdGF0ZT0iZG9uZSIsIG5vdGU9IlZpZGVvIHJlYWR5Iiwgc2l6ZT1zaXplLAogICAgICAgICAgICBkb3dubG9hZD1mIi9kb3dubG9hZC97amlkfS5tcDQiKQoKCmRlZiB3b3JrZXIoamlkLCBwYW5lbHMpOgogICAgdHJ5OgogICAgICAgIHJlbmRlcihqaWQsIHBhbmVscykKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBzZXRfam9iKGppZCwgc3RhdGU9ImVycm9yIiwgbm90ZT1zdHIoZSlbOjUwMF0pCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHNlcnZlciAtLS0KCkNPUlMgPSB7CiAgICAiQWNjZXNzLUNvbnRyb2wtQWxsb3ctT3JpZ2luIjogIioiLAogICAgIkFjY2Vzcy1Db250cm9sLUFsbG93LUhlYWRlcnMiOiAiY29udGVudC10eXBlLGF1dGhvcml6YXRpb24iLAogICAgIkFjY2Vzcy1Db250cm9sLUFsbG93LU1ldGhvZHMiOiAiR0VULFBPU1QsT1BUSU9OUyIsCn0KCgpjbGFzcyBIYW5kbGVyKEJhc2VIVFRQUmVxdWVzdEhhbmRsZXIpOgogICAgcHJvdG9jb2xfdmVyc2lvbiA9ICJIVFRQLzEuMSIKCiAgICBkZWYgbG9nX21lc3NhZ2Uoc2VsZiwgKmEpOgogICAgICAgIHBhc3MKCiAgICBkZWYgX3NlbmQoc2VsZiwgY29kZSwgb2JqLCBleHRyYT1Ob25lKToKICAgICAgICBib2R5ID0ganNvbi5kdW1wcyhvYmopLmVuY29kZSgpCiAgICAgICAgc2VsZi5zZW5kX3Jlc3BvbnNlKGNvZGUpCiAgICAgICAgZm9yIGssIHYgaW4geyoqQ09SUywgKiooZXh0cmEgb3Ige30pfS5pdGVtcygpOgogICAgICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKGssIHYpCiAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1UeXBlIiwgImFwcGxpY2F0aW9uL2pzb24iKQogICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoIkNvbnRlbnQtTGVuZ3RoIiwgc3RyKGxlbihib2R5KSkpCiAgICAgICAgc2VsZi5lbmRfaGVhZGVycygpCiAgICAgICAgc2VsZi53ZmlsZS53cml0ZShib2R5KQoKICAgIGRlZiBfYXV0aChzZWxmKToKICAgICAgICBpZiBub3QgVE9LRU46CiAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgcmV0dXJuIHNlbGYuaGVhZGVycy5nZXQoIkF1dGhvcml6YXRpb24iLCAiIikgPT0gZiJCZWFyZXIge1RPS0VOfSIKCiAgICBkZWYgZG9fT1BUSU9OUyhzZWxmKToKICAgICAgICBzZWxmLnNlbmRfcmVzcG9uc2UoMjA0KQogICAgICAgIGZvciBrLCB2IGluIENPUlMuaXRlbXMoKToKICAgICAgICAgICAgc2VsZi5zZW5kX2hlYWRlcihrLCB2KQogICAgICAgIHNlbGYuZW5kX2hlYWRlcnMoKQoKICAgIGRlZiBkb19HRVQoc2VsZik6CiAgICAgICAgcGF0aCA9IHVybHBhcnNlKHNlbGYucGF0aCkucGF0aAogICAgICAgIGlmIHBhdGggPT0gIi9oZWFsdGgiOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fc2VuZCgyMDAsIHsib2siOiBUcnVlLCAiZ3B1IjogTlZFTkMsICJsYW5lcyI6IExBTkVTfSkKICAgICAgICBtID0gcmUubWF0Y2gociJeL3N0YXR1cy8oW1x3LV0rKSQiLCBwYXRoKQogICAgICAgIGlmIG06CiAgICAgICAgICAgIGogPSBKT0JTLmdldChtLmdyb3VwKDEpKQogICAgICAgICAgICByZXR1cm4gc2VsZi5fc2VuZCgyMDAsIGopIGlmIGogZWxzZSBzZWxmLl9zZW5kKDQwNCwgeyJlcnJvciI6ICJubyBqb2IifSkKICAgICAgICBtID0gcmUubWF0Y2gociJeL2Rvd25sb2FkLyhbXHctXSspXC5tcDQkIiwgcGF0aCkKICAgICAgICBpZiBtOgogICAgICAgICAgICBmID0gb3MucGF0aC5qb2luKE9VVCwgZiJ7bS5ncm91cCgxKX0ubXA0IikKICAgICAgICAgICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKGYpOgogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYuX3NlbmQoNDA0LCB7ImVycm9yIjogIm5vdCByZWFkeSJ9KQogICAgICAgICAgICBzaXplID0gb3MucGF0aC5nZXRzaXplKGYpCiAgICAgICAgICAgIHNlbGYuc2VuZF9yZXNwb25zZSgyMDApCiAgICAgICAgICAgIGZvciBrLCB2IGluIENPUlMuaXRlbXMoKToKICAgICAgICAgICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoaywgdikKICAgICAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1UeXBlIiwgInZpZGVvL21wNCIpCiAgICAgICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoIkNvbnRlbnQtTGVuZ3RoIiwgc3RyKHNpemUpKQogICAgICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJDb250ZW50LURpc3Bvc2l0aW9uIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmJ2F0dGFjaG1lbnQ7IGZpbGVuYW1lPSJtYW5nYS12aWRlby5tcDQiJykKICAgICAgICAgICAgc2VsZi5lbmRfaGVhZGVycygpCiAgICAgICAgICAgIHdpdGggb3BlbihmLCAicmIiKSBhcyBmaDoKICAgICAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihmaCwgc2VsZi53ZmlsZSwgMTAyNCAqIDEwMjQpCiAgICAgICAgICAgIHJldHVybgogICAgICAgIHNlbGYuX3NlbmQoNDA0LCB7ImVycm9yIjogIm5vdCBmb3VuZCJ9KQoKICAgIGRlZiBkb19QT1NUKHNlbGYpOgogICAgICAgIGlmIHVybHBhcnNlKHNlbGYucGF0aCkucGF0aCAhPSAiL3JlbmRlciI6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9zZW5kKDQwNCwgeyJlcnJvciI6ICJub3QgZm91bmQifSkKICAgICAgICBpZiBub3Qgc2VsZi5fYXV0aCgpOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fc2VuZCg0MDEsIHsiZXJyb3IiOiAiYmFkIHRva2VuIn0pCiAgICAgICAgbiA9IGludChzZWxmLmhlYWRlcnMuZ2V0KCJDb250ZW50LUxlbmd0aCIsICIwIikpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBkYXRhID0ganNvbi5sb2FkcyhzZWxmLnJmaWxlLnJlYWQobikgb3IgYiJ7fSIpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX3NlbmQoNDAwLCB7ImVycm9yIjogImJhZCBqc29uIn0pCiAgICAgICAgcGFuZWxzID0gZGF0YS5nZXQoInBhbmVscyIpIG9yIFtdCiAgICAgICAgaWYgbm90IHBhbmVsczoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX3NlbmQoNDAwLCB7ImVycm9yIjogIm5vIHBhbmVscyJ9KQogICAgICAgIGppZCA9IHV1aWQudXVpZDQoKS5oZXhbOjEyXQogICAgICAgIHdpdGggTE9DSzoKICAgICAgICAgICAgSk9CU1tqaWRdID0geyJpZCI6IGppZCwgInN0YXRlIjogInJ1bm5pbmciLCAicGN0IjogMCwKICAgICAgICAgICAgICAgICAgICAgICAgICJub3RlIjogIlF1ZXVlZCIsICJwYW5lbHMiOiBsZW4ocGFuZWxzKX0KICAgICAgICB0aHJlYWRpbmcuVGhyZWFkKHRhcmdldD13b3JrZXIsIGFyZ3M9KGppZCwgcGFuZWxzKSwgZGFlbW9uPVRydWUpLnN0YXJ0KCkKICAgICAgICBzZWxmLl9zZW5kKDIwMCwgeyJpZCI6IGppZH0pCgoKZGVmIHNlcnZlKHBvcnQ9ODAwMCk6CiAgICBUaHJlYWRpbmdIVFRQU2VydmVyKCgiMC4wLjAuMCIsIHBvcnQpLCBIYW5kbGVyKS5zZXJ2ZV9mb3JldmVyKCkK"
    open('/content/encoder_server.py', 'wb').write(base64.b64decode(_B64))

import importlib.util
spec = importlib.util.spec_from_file_location('encoder_server', '/content/encoder_server.py')
enc = importlib.util.module_from_spec(spec); spec.loader.exec_module(enc)
threading.Thread(target=enc.serve, kwargs={'port': 8000}, daemon=True).start()
time.sleep(2)
print('encoder running · GPU NVENC =', enc.NVENC)

p = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000','--no-autoupdate'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = None
for line in p.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n' + '='*64)
print('PASTE THIS INTO THE APP:')
print(url or 'tunnel failed — re-run this cell')
print('='*64)


In [ ]:
#@title 2 · Start the encoder + public https tunnel
APP_URL = "https://script-to-epic.lovable.app"  #@param {type:"string"}
TOKEN   = ""  #@param {type:"string"}

import os, re, subprocess, threading, time, urllib.request
os.environ['SW_TOKEN'] = TOKEN
urllib.request.urlretrieve(APP_URL.rstrip('/') + '/colab/encoder_server.py', '/content/encoder_server.py')

import importlib.util
spec = importlib.util.spec_from_file_location('encoder_server', '/content/encoder_server.py')
enc = importlib.util.module_from_spec(spec); spec.loader.exec_module(enc)
threading.Thread(target=enc.serve, kwargs={'port': 8000}, daemon=True).start()
time.sleep(2)
print('encoder running · GPU NVENC =', enc.NVENC)

p = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000','--no-autoupdate'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = None
for line in p.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n' + '='*64)
print('PASTE THIS INTO THE APP:')
print(url or 'tunnel failed — re-run this cell')
print('='*64)

In [ ]:
#@title 3 · Keep alive (leave running while the video encodes)
import time
while True:
    time.sleep(60)